# 🛸 Autonomous Drone Security Analyst Agent Sandbox

Welcome to the **Drone Security Analyst Agent** interactive notebook. This notebook demonstrates the end-to-end multimodal architecture, including:
1. **Synchronized Ingestion**: Telemetry validation & clock alignment
2. **Structured VLM Inference**: Parsing aerial frames into typed entities & activities
3. **Stateful Event Tracking**: Cross-frame entity resolution (e.g. Ford F150 trajectory)
4. **Heuristic Rule Engine**: Real-time multi-factor risk & security alerts
5. **SQLite FTS5 Indexing**: High-speed hybrid semantic search
6. **Tool-Using Analyst Agent**: Conversational forensic Q&A

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on PYTHONPATH
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.storage.database import Database
from app.storage.repositories import FrameRepository, ObservationRepository, AlertRepository, EventRepository
from app.search.fts import FTSIndexer
from app.search.retriever import HybridRetriever
from app.pipeline.simulator import generate_simulated_packets
from app.pipeline.processor import FrameProcessor
from app.agent.tools import SecurityAgentTools
from app.agent.assistant import SecurityAnalystAssistant
from app.models.query import SearchRequest, ChatRequest

print("✅ All modules loaded successfully!")

## 1. Initialize Isolated Sandbox Database & Pipeline

In [ ]:
# Initialize a local sandbox database
sandbox_db = Database("../data/sandbox_drone.db")
frame_repo = FrameRepository(sandbox_db)
obs_repo = ObservationRepository(sandbox_db)
alert_repo = AlertRepository(sandbox_db)
event_repo = EventRepository(sandbox_db)
fts_indexer = FTSIndexer(sandbox_db)
retriever = HybridRetriever(sandbox_db, fts_indexer)
processor = FrameProcessor(frame_repo, obs_repo, alert_repo, event_repo)

print(f"Database ready at: {sandbox_db.db_path}")

## 2. Ingest Simulated Flight Feed & Execute VLM Pipeline

In [ ]:
packets = generate_simulated_packets()
print(f"Ingesting {len(packets)} simulated surveillance frames...\n")

for packet in packets:
    res = processor.process_frame(packet)
    print(f"Frame {packet.frame_id} @ {packet.telemetry.location_tag} [{packet.timestamp.strftime('%H:%M')}]:")
    print(f"  - Description : {res['observation']['description']}")
    print(f"  - Processing  : Total {res['metrics_ms']['total']}ms (VLM: {res['metrics_ms']['vlm']}ms, Index: {res['metrics_ms']['indexing']}ms)")
    if res['alerts_generated']:
        for alt in res['alerts_generated']:
            print(f"  🚨 ALERT [{alt['severity']}]: {alt['message']}")
    print("-" * 70)

## 3. Querying the FTS5 Full-Text Search Index

In [ ]:
# Search 1: Find all frames mentioning vehicles / trucks
truck_search = retriever.search(SearchRequest(query="truck F150"))
print(f"Search for 'truck F150': Found {truck_search.total_matches} matches")
for r in truck_search.results:
    print(f"  • Frame {r.frame_id} [{r.location}] (Rank: {r.rank_score}): {r.description}")

print("\n" + "="*50 + "\n")

# Search 2: Find loitering personnel near gates
person_search = retriever.search(SearchRequest(query="loitering hoodie", location="Main Gate"))
print(f"Search for 'loitering hoodie' at 'Main Gate': Found {person_search.total_matches} matches")
for r in person_search.results:
    print(f"  • Frame {r.frame_id} [{r.location}]: {r.description} | Alert: {r.alert_severity}")

## 4. Stateful Cross-Frame Tracked Events

In [ ]:
events = event_repo.get_all_events()
print(f"Total Multi-Frame Entity Profiles Tracked: {len(events)}\n")
for e in events:
    print(f"Entity: {e.label} [{e.entity_type.upper()}]")
    print(f"  - Occurrence Count : {e.occurrence_count}")
    print(f"  - Visited Locations: {' -> '.join(e.locations)}")
    print(f"  - First Seen       : {e.first_seen}")
    print(f"  - Last Seen        : {e.last_seen}")
    print("")

## 5. Conversational Security Analyst Agent (`/chat`)

In [ ]:
tools = SecurityAgentTools(retriever, frame_repo, alert_repo, event_repo)
agent = SecurityAnalystAssistant(tools)

queries = [
    "How many times was the blue Ford truck observed today?",
    "Was anyone loitering near the main gate during the night?",
    "What was the highest severity security incident recorded?",
    "Give me a summary of all activity today."
]

for q in queries:
    resp = agent.process_query(ChatRequest(prompt=q))
    print(f"🧑‍✈️ Operator Query : {q}")
    print(f"🤖 Analyst Agent  : {resp.response}")
    print(f"🛠️ Tools Executed : {[t.tool_name for t in resp.tools_used]}")
    print(f"🏷️ Intent Detected: {resp.intent} (Confidence: {resp.confidence})")
    print("-" * 70)